In [14]:
import duckdb
import pandas as pd
import os
from datetime import datetime
import sqlite3

In [28]:
# Sempre verifique se a variável já existe e feche-a antes de reconectar
if 'con' in globals():
    try:
        con.close()
    except:
        pass

# Agora conecte com segurança
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [22]:
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao 
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-21 07:56:12.225152
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-21 07:56:12.225152
2,10003,PREGO,BT10,100,0,z0019_2.csv,2026-05-21 07:56:12.225152


In [23]:
con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_produtos (
                NATBR VARCHAR,
                MAKTX VARCHAR,
                WERKS VARCHAR,
                MAINS VARCHAR,
                LABST VARCHAR,
                nome_arquivo VARCHAR,
                data_ingestao TIMESTAMP
            )
            """)

In [24]:
con.execute("insert into bronze_produtos select * from df")

In [31]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-20 15:23:13.616304
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-20 15:23:13.616304
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-05-20 15:23:13.616304
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-21 07:56:12.225152
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-21 07:56:12.225152


In [29]:
con.execute("alter table bronze_produtos rename to bronze_z0019")

In [32]:
con.close()